# Objective: solve a three-bus DCOPF fixture

Use a transparent economic-dispatch fixture to check balance, branch flows, generator limits, and objective accounting before scaling to a larger model.

In [ ]:
from pathlib import Path
import json, os, sys

TUTORIAL_ROOT = Path(os.environ.get('TUTORIAL_ROOT', Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('TUTORIAL_OUTPUT_ROOT', TUTORIAL_ROOT / 'outputs')).resolve(); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
demand = {'1': 30.0, '2': 20.0, '3': 10.0}
generators = {'g1': {'bus': '1', 'max_mw': 50.0, 'cost': 10.0}, 'g2': {'bus': '3', 'max_mw': 40.0, 'cost': 25.0}}
branches = [('1', '2', 0.5, 25.0), ('2', '3', 0.5, 25.0)]


In [ ]:
print('Environment report')
print({'python': sys.version.split()[0], 'solver': 'deterministic greedy fixture solve', 'fixture_first': True})


In [ ]:
total_demand = sum(demand.values())
g1 = min(generators['g1']['max_mw'], total_demand)
g2 = total_demand - g1
generation = {'g1': g1, 'g2': g2}
assert abs(sum(generation.values()) - total_demand) < 1e-9
assert all(0 <= generation[key] <= generators[key]['max_mw'] for key in generation)
flows = {'1-2': generation['g1'] - demand['1'], '2-3': generation['g1'] - demand['1'] - demand['2']}
assert all(abs(value) <= 25.0 for value in flows.values())
balance = {'bus_1': generation['g1'] - demand['1'] - flows['1-2'], 'bus_2': flows['1-2'] - demand['2'] - flows['2-3'], 'bus_3': generation['g2'] + flows['2-3'] - demand['3']}
assert all(abs(value) < 1e-9 for value in balance.values())
objective = generation['g1'] * 10.0 + generation['g2'] * 25.0
print('status: optimal_fixture_solution'); print('objective_usd_per_hour:', objective); print('generation_mw:', generation); print('branch_flows_mw:', flows); print('balance_mw:', balance)


In [ ]:
OUTPUT = OUTPUT_ROOT / 'three_bus_dcopf.json'
payload = {'status': 'optimal_fixture_solution', 'objective_usd_per_hour': objective, 'generation_mw': generation, 'branch_flows_mw': flows, 'balance_mw': balance, 'note': 'Educational fixture; not an ACTIVSg2000 runtime result.'}
OUTPUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('OUTPUT:', OUTPUT)
